# Preprocess the Inputs, Checking Every Step for Leakage

## Natural-language request

> Create another notebook where all the relevant preprocessing is done, with each and every step of preprocessing checked and rechecked for data leakage of any kind.

This notebook turns the eight inputs from notebook 08 into the numeric table a model needs. It **fits every step on train only**, applies the fitted steps unchanged to val, and checks for leakage **twice**: once right after each step, and again at the end against the saved files. **Test is never opened.**

## The kinds of leakage checked

| # | Leakage | How it could happen here | Where it is checked |
|---|---|---|---|
| **L1** | **Outcome in the inputs** | The label, an outcome column or a scoring-only column reaches the model. | Step 1, final recheck |
| **L2** | **Inputs not known on the forecast morning** | A column whose value is set or changed after morning M. | Decided by logic in notebook 06; **this notebook cannot test it**. Step 1 rechecks that the inputs are exactly notebook 06's, and names the four provisional ones. |
| **L3** | **Later periods shaping the fit** | Category lists, the rare-category cut-off or scaling statistics computed with val (or test) rows. | Steps 2 and 5, final recheck |
| **L4** | **The label shaping the preprocessing** | A step such as target encoding that uses `is_canceled` while fitting. | Step 6, final recheck |
| **L5** | **One row's output depending on other rows** | A transform that uses statistics of the batch it is given, so val rows would influence each other. | Step 7, final recheck |
| **L6** | **Identifiers or dates reaching the model** | The row number, `arrival_date`, `adr` or `nights` slipping into the output table. | Step 8, final recheck |
| **L7** | **Test used before the end** | Any test file read before every choice is frozen (project rule 2). | The loader below refuses to open test. |
| **L8** | **Hidden leakage showing as a perfect predictor** | A value that predicts the train label almost perfectly can be a sign of leakage. | Step 3: **reported for review only**, never used to choose anything. |

No check here can prove a column was known on morning M (L2). Our data is the final export, so that needs timestamped reservation snapshots (project rule 7).

## Setup

Loads train and val exactly as notebook 09 does, from the files notebook 08 wrote. `load_split` accepts only `"train"` and `"val"`, so this notebook cannot open test by mistake (L7).

The checks are written once, as functions, so the final cell can run every one of them again on the saved files.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# Find the repository so this works from the repo folder or notebooks/.
repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "contract.py").exists():
        repo_root = folder
        break
if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import contract

split_dir = repo_root / "data" / "splits"
if not (split_dir / "split_columns.json").exists():
    raise FileNotFoundError("data/splits/ is missing. Run notebooks/08_split_train_val_test.ipynb first.")
manifest = json.loads((split_dir / "split_columns.json").read_text(encoding="utf-8"))

inputs = manifest["inputs"]
provisional = manifest["provisional_inputs"]
label = manifest["label"]
scoring_only = manifest["scoring_only"]


def load_split(name):
    # L7: test stays closed until every choice is frozen.
    if name not in ("train", "val"):
        raise ValueError("This notebook only opens train and val.")
    return pd.read_csv(split_dir / f"{name}.csv", index_col="row",
                       parse_dates=["arrival_date"], dtype={"agent": str, "company": str})


train = load_split("train")
val = load_split("val")
X_train, y_train = train[inputs], train[label]
X_val, y_val = val[inputs], val[label]

checks = []  # one row per check that has run: step, leakage type, what was checked


def passed(step, leakage, what):
    checks.append({"step": step, "leakage": leakage, "check": what, "result": "PASS"})
    print(f"PASS  [{leakage}] {what}")


print(f"train {X_train.shape}, val {X_val.shape}; inputs: {inputs}")

train (49157, 8), val (29546, 8); inputs: ['hotel', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'agent', 'company']


## Step 1: recheck the inputs (L1, L2, L6)

The inputs must be **exactly** the columns notebook 06 marked `input` or `provisional input`, read again here from notebook 06 itself, not only from the manifest. Then:

- no outcome column (`is_canceled`, `reservation_status`, `reservation_status_date`) is an input;
- no scoring-only column (`arrival_date`, `adr`, `nights`) and no identifier is an input;
- the provisional inputs are the ones notebook 06 marks provisional.

In [2]:
outcome_columns = ["is_canceled", "reservation_status", "reservation_status_date"]
forbidden = set(outcome_columns) | set(scoring_only) | {"row", "arrival_date_year"}


def check_inputs(columns):
    notebook_06 = json.loads((repo_root / "notebooks" / "06_forecast_column_selection.ipynb").read_text(encoding="utf-8"))
    my_decisions = {}
    for cell in notebook_06["cells"]:
        if cell.get("id", "").startswith("decide-"):
            exec("".join(cell["source"]), {"my_decisions": my_decisions})
    decided_inputs = {c for c, d in my_decisions.items() if d["use"] in ("input", "provisional input")}
    decided_provisional = {c for c, d in my_decisions.items() if d["use"] == "provisional input"}

    assert set(columns) == decided_inputs, f"inputs differ from notebook 06: {set(columns) ^ decided_inputs}"
    passed("1", "L2", "inputs are exactly notebook 06's input and provisional-input columns")
    assert set(provisional) == decided_provisional
    passed("1", "L2", f"provisional inputs match notebook 06 ({len(provisional)}): timing still unverified")
    assert not set(columns) & forbidden, f"forbidden columns among inputs: {set(columns) & forbidden}"
    passed("1", "L1", "no label, outcome, scoring-only or identifier column is an input")
    for name, X in {"train": X_train, "val": X_val}.items():
        assert list(X.columns) == list(columns)
    passed("1", "L6", "X_train and X_val hold the inputs and nothing else")


check_inputs(inputs)

PASS  [L2] inputs are exactly notebook 06's input and provisional-input columns
PASS  [L2] provisional inputs match notebook 06 (4): timing still unverified
PASS  [L1] no label, outcome, scoring-only or identifier column is an input
PASS  [L6] X_train and X_val hold the inputs and nothing else


## Step 2: recheck the periods (L3)

Notebook 08 already checked the split. It is rechecked here because everything below relies on it: every train arrival comes before every val arrival, and no booking is in both.

In [3]:
def check_periods(train_part, val_part):
    assert train_part["arrival_date"].max() < pd.Timestamp(contract.TRAIN_END) <= val_part["arrival_date"].min()
    assert val_part["arrival_date"].max() < pd.Timestamp(contract.PROOF_START)
    passed("2", "L3", f"train arrivals end before {contract.TRAIN_END}; val arrivals run from {contract.TRAIN_END} to before {contract.PROOF_START}")
    assert not set(train_part.index) & set(val_part.index)
    passed("2", "L3", "no booking is in both train and val")


check_periods(train, val)

PASS  [L3] train arrivals end before 2016-07-01; val arrivals run from 2016-07-01 to before 2017-01-01
PASS  [L3] no booking is in both train and val


## Step 3: scan train for near-perfect predictors (L8, review only)

A value shared by many bookings that cancel **all** or **none** of the time can be a sign that a column records something that happened after booking. This scan uses **train only** and lists every input value held by at least 100 train bookings whose cancellation rate is at least 99% or at most 1%.

**This does not choose or drop anything** (project rule 2, and notebook 07, P4): a strong predictor is not proof of leakage, and a weak one is not proof of safety. Anything listed needs an explanation from how the hotel records that value.

In [4]:
flags = []
for column in inputs:
    by_value = train.groupby(column)[label].agg(bookings="size", cancellation_rate="mean")
    extreme = by_value[(by_value["bookings"] >= 100)
                       & ((by_value["cancellation_rate"] >= 0.99) | (by_value["cancellation_rate"] <= 0.01))]
    for value, row in extreme.iterrows():
        flags.append({"column": column, "value": value, "train bookings": int(row["bookings"]),
                      "cancellation rate": f"{row['cancellation_rate']:.1%}"})

if flags:
    print(f"{len(flags)} value(s) to review (nothing is removed or changed):")
    display(pd.DataFrame(flags))
else:
    print("No input value with 100+ train bookings cancels at least 99% or at most 1% of the time.")

1 value(s) to review (nothing is removed or changed):


,column,value,train bookings,cancellation rate
0,agent,236,159,100.0%


## Step 4: define the preprocessing

Each input gets one treatment. Every number a step needs (category lists, which categories are rare, means and spreads) is learnt **from train only** in Step 5.

| Input | Treatment | Why |
|---|---|---|
| `hotel` | One 0/1 column (`Resort Hotel` = 1) | Two values; one column carries all the information. |
| `market_segment`, `distribution_channel`, `agent`, `company` | One 0/1 column per category. Categories with fewer than **30 train bookings** share one "infrequent" column, and so does **any value never seen in train**. | Rare categories give too few bookings to learn from; val and test hold agents and companies that train never saw (notebook 08, Step 5). |
| `previous_cancellations`, `previous_bookings_not_canceled` | `log(1 + count)`, then centred and scaled with the **train** mean and standard deviation | The counts have long tails (train up to 26 and 36, val up to 58), and scaling puts them on the same footing as the 0/1 columns for logistic regression. |
| `is_repeated_guest` | Kept as 0/1 | Already a 0/1 column. |

**The 30-booking cut-off is a starting setting, not a tuned value.** Changing it is a modelling choice and must be compared on val, never on test (project rule 2).

Nothing here uses the label. There is **no target encoding**, which would be the main way preprocessing could leak the outcome (L4).

In [5]:
MIN_BOOKINGS = 30
category_inputs = ["hotel", "market_segment", "distribution_channel", "agent", "company"]
count_inputs = ["previous_cancellations", "previous_bookings_not_canceled"]
flag_inputs = ["is_repeated_guest"]
assert sorted(category_inputs + count_inputs + flag_inputs) == sorted(inputs), "every input needs exactly one treatment"


def make_preprocessor():
    # A fresh, unfitted preprocessor. Nothing is learnt until .fit() is called on train.
    return ColumnTransformer(
        transformers=[
            ("category", OneHotEncoder(min_frequency=MIN_BOOKINGS, handle_unknown="infrequent_if_exist",
                                       drop="if_binary", sparse_output=False), category_inputs),
            ("count", Pipeline([("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                                ("scale", StandardScaler())]), count_inputs),
            ("flag", "passthrough", flag_inputs),
        ],
        remainder="drop",
    ).set_output(transform="pandas")


# Values never seen in train are expected (Step 9 counts them), so silence the encoder's warning about them.
warnings.filterwarnings("ignore", message="Found unknown categories")
print(make_preprocessor())

ColumnTransformer(transformers=[('category',
                                 OneHotEncoder(drop='if_binary',
                                               handle_unknown='infrequent_if_exist',
                                               min_frequency=30,
                                               sparse_output=False),
                                 ['hotel', 'market_segment',
                                  'distribution_channel', 'agent', 'company']),
                                ('count',
                                 Pipeline(steps=[('log1p',
                                                  FunctionTransformer(feature_names_out='one-to-one',
                                                                      func=<ufunc 'log1p'>)),
                                                 ('scale', StandardScaler())]),
                                 ['previous_cancellations',
                                  'previous_bookings_not_canceled']),
                       

## Step 5: fit on train only, then transform train and val (L3)

`preprocessor.fit(X_train)` is the **only** fit in this notebook. Val is only ever passed to `.transform()`, which applies what was learnt from train.

The checks then recompute, **by hand from train alone**, everything the preprocessor learnt and confirm it matches:

1. every category it knows appears in train;
2. the categories it treats as rare are exactly those with fewer than 30 **train** bookings;
3. the scaling mean and standard deviation equal the **train** values;
4. transforming val leaves the fitted preprocessor unchanged.

A check that cannot fail proves nothing, so the last check also refits on train + val and confirms the scaling statistics **would** change. That shows the check above would catch val leaking into the fit.

In [6]:
preprocessor = make_preprocessor().fit(X_train)
train_prepared = preprocessor.transform(X_train)
val_prepared = preprocessor.transform(X_val)


def check_fit_uses_train_only(fitted, X_fit):
    encoder = fitted.named_transformers_["category"]
    for column, known, rare in zip(category_inputs, encoder.categories_, encoder.infrequent_categories_):
        train_counts = X_fit[column].value_counts()
        assert set(known) == set(train_counts.index), f"{column}: categories are not exactly train's"
        expected_rare = set(train_counts[train_counts < MIN_BOOKINGS].index)
        assert set(rare if rare is not None else []) == expected_rare, f"{column}: rare categories differ from train counts"
    passed("5", "L3", "category lists and rare categories equal those counted in train alone")

    scaler = fitted.named_transformers_["count"].named_steps["scale"]
    logged = np.log1p(X_fit[count_inputs])
    assert np.allclose(scaler.mean_, logged.mean()) and np.allclose(scaler.scale_, logged.std(ddof=0))
    passed("5", "L3", "scaling mean and standard deviation equal train's")


def check_transform_does_not_refit(fitted, X_other):
    before = joblib.hash(fitted)
    fitted.transform(X_other)
    assert joblib.hash(fitted) == before
    passed("5", "L3", "transforming val leaves the fitted preprocessor unchanged")


check_fit_uses_train_only(preprocessor, X_train)
check_transform_does_not_refit(preprocessor, X_val)

# Sensitivity: fitting on train + val would give different statistics, so the checks above can detect it.
contaminated = make_preprocessor().fit(pd.concat([X_train, X_val]))
train_mean = preprocessor.named_transformers_["count"].named_steps["scale"].mean_
contaminated_mean = contaminated.named_transformers_["count"].named_steps["scale"].mean_
assert not np.allclose(train_mean, contaminated_mean)
passed("5", "L3", "a fit on train + val gives different statistics, so the checks above would catch it")

print(f"\ntrain_prepared {train_prepared.shape}, val_prepared {val_prepared.shape}")

PASS  [L3] category lists and rare categories equal those counted in train alone
PASS  [L3] scaling mean and standard deviation equal train's
PASS  [L3] transforming val leaves the fitted preprocessor unchanged


PASS  [L3] a fit on train + val gives different statistics, so the checks above would catch it

train_prepared (49157, 143), val_prepared (29546, 143)


## Step 6: the label plays no part (L4)

If any step used `is_canceled` while fitting, giving it a **shuffled** label would change the result. The check fits three fresh preprocessors on train: without a label, with the true label, and with the label shuffled. All three must produce **identical** val output.

In [7]:
def check_label_not_used(X_fit, y_fit, X_other):
    shuffled = pd.Series(np.random.default_rng(0).permutation(y_fit.to_numpy()), index=y_fit.index)
    outputs = [make_preprocessor().fit(X_fit).transform(X_other),
               make_preprocessor().fit(X_fit, y_fit).transform(X_other),
               make_preprocessor().fit(X_fit, shuffled).transform(X_other)]
    assert outputs[0].equals(outputs[1]) and outputs[0].equals(outputs[2])
    passed("6", "L4", "fitting without, with, or with a shuffled label gives identical output")


check_label_not_used(X_train, y_train, X_val)

PASS  [L4] fitting without, with, or with a shuffled label gives identical output


## Step 7: each row is transformed on its own (L5)

A booking's prepared values must depend only on that booking and on what was learnt from train, never on which other bookings are transformed alongside it. The check transforms val in one go, in five separate pieces, and in reverse order, and requires identical results row by row.

In [8]:
def check_rowwise(fitted, X_other):
    whole = fitted.transform(X_other)
    bounds = np.linspace(0, len(X_other), 6, dtype=int)
    pieces = pd.concat([fitted.transform(X_other.iloc[start:end]) for start, end in zip(bounds[:-1], bounds[1:])])
    reversed_order = fitted.transform(X_other.iloc[::-1]).loc[X_other.index]
    assert whole.equals(pieces) and whole.equals(reversed_order)
    passed("7", "L5", "val output is the same whole, in pieces, or in reverse order")


check_rowwise(preprocessor, X_val)

PASS  [L5] val output is the same whole, in pieces, or in reverse order


## Step 8: check the prepared tables (L6)

- Every output column comes from one of the eight inputs; none from the label, the scoring-only columns or the row number.
- Train and val have the same columns in the same order, and the original row numbers are kept as the index, so each row can be traced back to its booking.
- There are no missing or infinite values.

In [9]:
def check_outputs(fitted, prepared):
    names = list(fitted.get_feature_names_out())
    for name in names:
        source = name.split("__", 1)[1]
        assert any(source == c or source.startswith(c + "_") for c in inputs), f"{name} does not come from an input"
        assert not any(source.startswith(c) for c in forbidden), f"{name} comes from a forbidden column"
    passed("8", "L6", f"all {len(names)} output columns come from the eight inputs")
    for name, (frame, raw) in prepared.items():
        assert list(frame.columns) == names and frame.index.equals(raw.index)
        values = frame.to_numpy(dtype=float)
        assert np.isfinite(values).all()
    passed("8", "L6", "train and val share the same columns, keep their row numbers, and have no missing or infinite values")


check_outputs(preprocessor, {"train": (train_prepared, X_train), "val": (val_prepared, X_val)})

# Output columns per input.
per_input = pd.Series([n.split("__", 1)[1] for n in preprocessor.get_feature_names_out()])
display(pd.Series({c: int(per_input.map(lambda s, c=c: s == c or s.startswith(c + "_")).sum()) for c in inputs},
                  name="output columns"))

PASS  [L6] all 143 output columns come from the eight inputs


PASS  [L6] train and val share the same columns, keep their row numbers, and have no missing or infinite values


hotel                               1
market_segment                      8
distribution_channel                5
is_repeated_guest                   1
previous_cancellations              1
previous_bookings_not_canceled      1
agent                             106
company                            20
Name: output columns, dtype: int64

## Step 9: how many bookings fall into the "infrequent" column

The share of bookings in each "infrequent" column, in train and in val. In val it also holds every value train never saw. A much larger share in val than in train means the model will know less about those bookings. This reads feature values only, and it chooses nothing.

In [10]:
rows = []
for column in category_inputs:
    name = f"category__{column}_infrequent_sklearn"
    if name in train_prepared:
        rows.append({"input": column,
                     "train: share infrequent": f"{train_prepared[name].mean():.1%}",
                     "val: share infrequent (rare or never seen)": f"{val_prepared[name].mean():.1%}",
                     "val: values never seen in train": f"{(~X_val[column].isin(set(X_train[column]))).mean():.1%}"})
display(pd.DataFrame(rows).set_index("input"))

,train: share infrequent,val: share infrequent (rare or never seen),val: values never seen in train
input,,,
market_segment,0.0%,0.0%,0.0%
distribution_channel,0.0%,0.0%,0.0%
agent,2.4%,4.3%,1.7%
company,2.2%,2.6%,1.1%


## Step 10: save the fitted preprocessor and the prepared tables

Writes to `data/preprocessed/` (inside `data/`, which `.gitignore` keeps out of the repository):

| File | Contents |
|---|---|
| `preprocessor.joblib` | The preprocessor fitted on train. Use it to transform val, and test **once** at the end. Never refit it on val or test. |
| `train_X.csv`, `val_X.csv` | The prepared inputs, with `row` (the booking's row number) as the first column. The label stays in `data/splits/`; join on `row`. |
| `feature_names.json` | The output column names, the inputs and the 30-booking setting. |

In [11]:
out_dir = repo_root / "data" / "preprocessed"
out_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(preprocessor, out_dir / "preprocessor.joblib")
train_prepared.to_csv(out_dir / "train_X.csv", index_label="row")
val_prepared.to_csv(out_dir / "val_X.csv", index_label="row")
(out_dir / "feature_names.json").write_text(json.dumps({
    "inputs": inputs,
    "provisional_inputs": provisional,
    "min_bookings_per_category": MIN_BOOKINGS,
    "fitted_on": "train only (arrivals before " + contract.TRAIN_END + ")",
    "output_columns": list(preprocessor.get_feature_names_out()),
}, indent=2) + "\n", encoding="utf-8")

for path in sorted(out_dir.iterdir()):
    print(f"Saved data/preprocessed/{path.name}")

Saved data/preprocessed/feature_names.json
Saved data/preprocessed/preprocessor.joblib
Saved data/preprocessed/train_X.csv
Saved data/preprocessed/val_X.csv


## Final recheck: run every check again on the saved files

Reloads the saved preprocessor and the prepared CSV files, and reruns **every** check from Steps 1 to 8 against them, plus three more:

- the reloaded preprocessor gives exactly the saved val table;
- the saved CSV files match the tables in memory;
- refitting from scratch on train gives the same preprocessor (the result is reproducible).

Any failure stops the cell.

In [12]:
checks.clear()
reloaded = joblib.load(out_dir / "preprocessor.joblib")
train_saved = pd.read_csv(out_dir / "train_X.csv", index_col="row")
val_saved = pd.read_csv(out_dir / "val_X.csv", index_col="row")

check_inputs(inputs)
check_periods(load_split("train"), load_split("val"))
check_fit_uses_train_only(reloaded, X_train)
check_transform_does_not_refit(reloaded, X_val)
check_label_not_used(X_train, y_train, X_val)
check_rowwise(reloaded, X_val)
check_outputs(reloaded, {"train": (train_saved, X_train), "val": (val_saved, X_val)})

assert np.allclose(reloaded.transform(X_val).to_numpy(dtype=float), val_saved.to_numpy(dtype=float))
passed("final", "L3", "the reloaded preprocessor reproduces the saved val table")
assert np.allclose(train_saved.to_numpy(dtype=float), train_prepared.to_numpy(dtype=float))
assert np.allclose(val_saved.to_numpy(dtype=float), val_prepared.to_numpy(dtype=float))
passed("final", "L6", "the saved CSV files match the prepared tables")
assert make_preprocessor().fit(X_train).transform(X_val).equals(reloaded.transform(X_val))
passed("final", "L3", "refitting on train from scratch gives the same result")

print()
display(pd.DataFrame(checks))
print(f"All {len(checks)} checks pass. L2 (whether each input was known on morning M) cannot be checked from this data; "
      f"{len(provisional)} inputs remain provisional: {provisional}.")

PASS  [L2] inputs are exactly notebook 06's input and provisional-input columns
PASS  [L2] provisional inputs match notebook 06 (4): timing still unverified
PASS  [L1] no label, outcome, scoring-only or identifier column is an input
PASS  [L6] X_train and X_val hold the inputs and nothing else


PASS  [L3] train arrivals end before 2016-07-01; val arrivals run from 2016-07-01 to before 2017-01-01
PASS  [L3] no booking is in both train and val
PASS  [L3] category lists and rare categories equal those counted in train alone
PASS  [L3] scaling mean and standard deviation equal train's
PASS  [L3] transforming val leaves the fitted preprocessor unchanged


PASS  [L4] fitting without, with, or with a shuffled label gives identical output


PASS  [L5] val output is the same whole, in pieces, or in reverse order
PASS  [L6] all 143 output columns come from the eight inputs
PASS  [L6] train and val share the same columns, keep their row numbers, and have no missing or infinite values


PASS  [L3] the reloaded preprocessor reproduces the saved val table


PASS  [L6] the saved CSV files match the prepared tables


PASS  [L3] refitting on train from scratch gives the same result



,step,leakage,check,result
0,1,L2,inputs are exactly notebook 06's input and pro...,PASS
1,1,L2,provisional inputs match notebook 06 (4): timi...,PASS
2,1,L1,"no label, outcome, scoring-only or identifier ...",PASS
3,1,L6,X_train and X_val hold the inputs and nothing ...,PASS
4,2,L3,train arrivals end before 2016-07-01; val arri...,PASS
5,2,L3,no booking is in both train and val,PASS
6,5,L3,category lists and rare categories equal those...,PASS
7,5,L3,scaling mean and standard deviation equal train's,PASS
8,5,L3,transforming val leaves the fitted preprocesso...,PASS
9,6,L4,"fitting without, with, or with a shuffled labe...",PASS


All 16 checks pass. L2 (whether each input was known on morning M) cannot be checked from this data; 4 inputs remain provisional: ['is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'company'].


## Summary

- **Preprocessing is fitted on train only** and applied unchanged to val. Category lists, rare categories and scaling statistics were recomputed by hand from train and match. A fit that included val would have been caught.
- **The label plays no part** in preprocessing. A shuffled label gives identical output, and there is no target encoding.
- **Each booking is transformed on its own**, and only the eight inputs reach the output. No identifier, date, price or outcome does.
- **Test was never opened.**
- **Every check ran twice**: after its step, and again on the saved files.

**What this notebook cannot rule out:**

- **Timing leakage (L2).** Whether each input was known on the forecast morning is a judgment from the source definitions, made in notebook 06. Four inputs are **provisional**: `is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled` and `company`. Every result that uses them must say their timing is unverified.
- **Step 3's flags.** Any value listed there needs an explanation from how the hotel records it before results are trusted.

**Next:** fit the flat-rule baseline and candidate models on `train_X.csv` with the train labels, predict val with the saved preprocessor, and score val with `contract.nightly_table` and `contract.night_cost` (project rules 5 and 6).